In [50]:
import numpy as np
import pandas as pd
import os
import openpyxl
import re

In [51]:
data_path = os.path.join(os.getcwd(), "..", "data", "training")
raw_data_path = os.path.join(data_path, "raw")
converted_data_path = os.path.join(data_path, "converted")
os.makedirs(converted_data_path, exist_ok=True)

In [52]:
raw_files = os.listdir(raw_data_path)
raw_files = [file for file in raw_files if file.endswith(".xlsx") and not file.startswith("~$")]
raw_files

['raw.xlsx', 'raw2.xlsx']

In [53]:
file_mapping = dict()
for file in raw_files:
    num = re.search(r".*(\d+).*", file.lower(), flags=re.IGNORECASE)
    num = num.group(1) if num else "1"
    name = f'series_{num}'
    file_mapping[file] = name
    print(f"Mapped {file} to {name}.")

file_mapping

Mapped raw.xlsx to series_1.
Mapped raw2.xlsx to series_2.


{'raw.xlsx': 'series_1', 'raw2.xlsx': 'series_2'}

In [54]:
excel_files: list[tuple[str, pd.ExcelFile | None]] = [("", None) for _ in range(len(raw_files))]
for i, file in enumerate(raw_files):
    print(f"Loading {file}...")
    excel_files[i] = (file, pd.ExcelFile(os.path.join(raw_data_path, file), engine="openpyxl", engine_kwargs={"read_only": True}))
    print(f"Loaded {file} successfully.")

Loading raw.xlsx...
Loaded raw.xlsx successfully.
Loading raw2.xlsx...
Loaded raw2.xlsx successfully.
Loaded raw.xlsx successfully.
Loading raw2.xlsx...
Loaded raw2.xlsx successfully.


In [55]:
sheet_lists: list[tuple[str, list[str]]] = [(wb[0], [str(sheet) for sheet in wb[1].sheet_names]) for wb in excel_files if wb[1] is not None]
sheet_lists

[('raw.xlsx',
  ['Filtered',
   '+ Annotation',
   'Messwerte Wurst 1, 1. Messreihe',
   'Messwerte Wurst 2, 1. Messreihe',
   'Messwerte Wurst 3, 1. Messreihe',
   'Messwerte Wurst 4, 1. Messreihe',
   'Messwerte Wurst 5, 1. Messreihe']),
 ('raw2.xlsx', ['Alle', 'Wurst1', 'Wurst2', 'Wurst3', 'Wurst4', 'Wurst5'])]

In [56]:
target_sheets = []
for (file, sheet_list) in sheet_lists:
    for sheet in sheet_list:
        if re.match(r".*alle.*", sheet.lower(), re.IGNORECASE) or re.match(r".*annotation.*", sheet.lower(), re.IGNORECASE):
            target_sheets.append(sheet)

target_sheets

['+ Annotation', 'Alle']

In [57]:
sensor_mapping = {
    "WS202": r".*milesight.*",
    "LSE01": r".*wurst[^\d]*(\d+).*feuchtigkeit.*",
    "LTC-FS": r".*wurst[^\d]*(\d+).*temperatur.*"
}
sausage_sensors = set(sensor_mapping.keys()) - {"WS202"}
environment_sensors = set(sensor_mapping.keys()) - sausage_sensors
sausage_sensors = set([sensor_mapping[sensor] for sensor in sausage_sensors])
environment_sensors = set([sensor_mapping[sensor] for sensor in environment_sensors])
sensor_mapping, sausage_sensors, environment_sensors

({'WS202': '.*milesight.*',
  'LSE01': '.*wurst[^\\d]*(\\d+).*feuchtigkeit.*',
  'LTC-FS': '.*wurst[^\\d]*(\\d+).*temperatur.*'},
 {'.*wurst[^\\d]*(\\d+).*feuchtigkeit.*',
  '.*wurst[^\\d]*(\\d+).*temperatur.*'},
 {'.*milesight.*'})

In [58]:
dataframes = {}
for file, sheet_list in sheet_lists:
    sheet_list = [sheet for sheet in sheet_list if sheet in target_sheets]
    for sheet in sheet_list:
        print(f"Loading {sheet} in {file}...")
        df = pd.read_excel(os.path.join(raw_data_path, file), sheet_name=sheet
                            , engine="openpyxl", engine_kwargs={"read_only": True})
        dataframes[(file, sheet)] = df
        print(f"Loaded {sheet} in {file} successfully.")

Loading + Annotation in raw.xlsx...
Loaded + Annotation in raw.xlsx successfully.
Loading Alle in raw2.xlsx...
Loaded Alle in raw2.xlsx successfully.


In [59]:
sensor_targets = dict()
for file, sheet_list in sheet_lists:
    sheet_list = [sheet for sheet in sheet_list if sheet in target_sheets]
    for sheet in sheet_list:
        print(f"Processing {sheet} in {file}")
        df = dataframes.get((file, sheet))
        if df is None:
            raise ValueError(f"Dataframe for {sheet} in {file} not loaded.")
        distinct_measurement_names = df["Gerät"].unique().tolist()
        sausage_measurement_names = [name for name in distinct_measurement_names if any(re.match(pattern, name.lower(), re.IGNORECASE) for pattern in sausage_sensors)]
        environment_measurement_names = [name for name in distinct_measurement_names if any(re.match(pattern, name.lower(), re.IGNORECASE) for pattern in environment_sensors)]
        print(f"Found distinct measurement names in {sheet} of {file}: {distinct_measurement_names}")
        print(f"Identified sausage measurement names in {sheet} of {file}: {sausage_measurement_names}")
        print(f"Identified environment measurement names in {sheet} of {file}: {environment_measurement_names}")
        sausage_measurements = dict()
        sensor_files = dict()
        for measurement_name in sausage_measurement_names:                
            for sensor_name, pattern in sensor_mapping.items():
                re_match = re.match(pattern, measurement_name.lower(), re.IGNORECASE)
                if re_match:
                    sausage_measurements[measurement_name] = sensor_name
                    sensor_files[measurement_name] = f"{file_mapping[file]}_sausage_{re_match.group(1) if re_match.groups() else ''}"
                    print(f"Mapped measurement name {measurement_name} to sensor {sensor_name} in {sheet} of {file}.")
                    break
            else:
                print(f"No mapping found for measurement name {measurement_name} in {sheet} of {file}.")
        environment_measurements = dict()
        for measurement_name in environment_measurement_names:
            for sensor_name, pattern in sensor_mapping.items():
                re_match = re.match(pattern, measurement_name.lower(), re.IGNORECASE)
                if re_match:
                    environment_measurements[measurement_name] = sensor_name
                    print(f"Mapped measurement name {measurement_name} to sensor {sensor_name} in {sheet} of {file}.")
                    break
            else:
                print(f"No mapping found for measurement name {measurement_name} in {sheet} of {file}.")
        
        sausage_measurements = {k: sausage_measurements[k] for k in sorted(sausage_measurements.keys())}
        environment_measurements = {k: environment_measurements[k] for k in sorted(environment_measurements.keys())}
        sensor_files = {k: sensor_files[k] for k in sorted(sensor_files.keys())}
        print(f"Target measurements in {sheet} of {file}: {sausage_measurements}")
        print(f"Environment measurements in {sheet} of {file}: {environment_measurements}")
        print(f"Sensor files in {sheet} of {file}: {sensor_files}")
        sensor_targets[(file, sheet)] = (sausage_measurements, environment_measurements, sensor_files)

sensor_targets

Processing + Annotation in raw.xlsx
Found distinct measurement names in + Annotation of raw.xlsx: ['Wurst 3 Temperatur', 'Wurst 1 Feuchtigkeit, Temperatur & Leitwert', 'Wurst 3 PH-Wert', 'Wurst 4 PH-Wert', 'Wurst 4 Feuchtigkeit, Temperatur & Leitwert', 'Wurst 5 Feuchtigkeit, Temperatur & Leitwert', 'Wurst 3 Feuchtigkeit, Temperatur & Leitwert', 'Wurst 2 Temperatur', 'Wurst 5 Temperatur', 'Wurst 2 PH-Wert', 'Wurst 1 Temperatur', 'Umgebung Temperatur & Luftfeuchtigkeit', 'Wurst 4 Temperatur', 'Wurst 2 Feuchtigkeit, Temperatur & Leitwert', 'Wurst 5 PH-Wert', 'Wurst 1 PH-Wert', 'Milesight Umgebung PIR & Lichtintensität']
Identified sausage measurement names in + Annotation of raw.xlsx: ['Wurst 3 Temperatur', 'Wurst 1 Feuchtigkeit, Temperatur & Leitwert', 'Wurst 4 Feuchtigkeit, Temperatur & Leitwert', 'Wurst 5 Feuchtigkeit, Temperatur & Leitwert', 'Wurst 3 Feuchtigkeit, Temperatur & Leitwert', 'Wurst 2 Temperatur', 'Wurst 5 Temperatur', 'Wurst 1 Temperatur', 'Wurst 4 Temperatur', 'Wurst 2 F

{('raw.xlsx',
  '+ Annotation'): ({'Wurst 1 Feuchtigkeit, Temperatur & Leitwert': 'LSE01',
   'Wurst 1 Temperatur': 'LTC-FS',
   'Wurst 2 Feuchtigkeit, Temperatur & Leitwert': 'LSE01',
   'Wurst 2 Temperatur': 'LTC-FS',
   'Wurst 3 Feuchtigkeit, Temperatur & Leitwert': 'LSE01',
   'Wurst 3 Temperatur': 'LTC-FS',
   'Wurst 4 Feuchtigkeit, Temperatur & Leitwert': 'LSE01',
   'Wurst 4 Temperatur': 'LTC-FS',
   'Wurst 5 Feuchtigkeit, Temperatur & Leitwert': 'LSE01',
   'Wurst 5 Temperatur': 'LTC-FS'}, {'Milesight Umgebung PIR & Lichtintensität': 'WS202'}, {'Wurst 1 Feuchtigkeit, Temperatur & Leitwert': 'series_1_sausage_1',
   'Wurst 1 Temperatur': 'series_1_sausage_1',
   'Wurst 2 Feuchtigkeit, Temperatur & Leitwert': 'series_1_sausage_2',
   'Wurst 2 Temperatur': 'series_1_sausage_2',
   'Wurst 3 Feuchtigkeit, Temperatur & Leitwert': 'series_1_sausage_3',
   'Wurst 3 Temperatur': 'series_1_sausage_3',
   'Wurst 4 Feuchtigkeit, Temperatur & Leitwert': 'series_1_sausage_4',
   'Wurst 4 Tem

In [74]:
file_dfs: dict[str, pd.DataFrame] = dict()

for (file, sheet), (sausage_measurements, environment_measurements, sensor_files) in sensor_targets.items():
    print(f"Processing {sheet} in {file} for conversion...")
    df = dataframes.get((file, sheet))
    if df is None:
        raise ValueError(f"Dataframe for {sheet} in {file} not loaded.")
    
    for file_name in sensor_files.values():
        file_dfs[file_name] = pd.DataFrame()

    # Process sausage measurements
    for measurement_name, sensor_name in sausage_measurements.items():
        sensor_file = sensor_files.get(measurement_name)
        if sensor_file is None:
            print(f"No sensor file mapping found for measurement name {measurement_name} in {sheet} of {file}. Skipping conversion for this measurement.")
            continue
        print(f"Converting measurement {measurement_name} to sensor {sensor_name} and saving to {sensor_file}.")
        filtered_df = df.loc[df["Gerät"] == measurement_name]
        filtered_df["Gerät"] = sensor_name
        file_dfs[sensor_file] = pd.concat([file_dfs[sensor_file], filtered_df], ignore_index=True)

    # Process environment measurements
    for measurement_name, sensor_name in environment_measurements.items():
        print(f"Converting measurement {measurement_name} to sensor {sensor_name} and saving to environment file.")
        filtered_df = df.loc[df["Gerät"] == measurement_name]
        filtered_df["Gerät"] = sensor_name
        for file_name in file_dfs.keys():
            file_dfs[file_name] = pd.concat([file_dfs[file_name], filtered_df], ignore_index=True)

file_dfs

Processing + Annotation in raw.xlsx for conversion...
Converting measurement Wurst 1 Feuchtigkeit, Temperatur & Leitwert to sensor LSE01 and saving to series_1_sausage_1.
Converting measurement Wurst 1 Temperatur to sensor LTC-FS and saving to series_1_sausage_1.
Converting measurement Wurst 2 Feuchtigkeit, Temperatur & Leitwert to sensor LSE01 and saving to series_1_sausage_2.
Converting measurement Wurst 2 Temperatur to sensor LTC-FS and saving to series_1_sausage_2.
Converting measurement Wurst 3 Feuchtigkeit, Temperatur & Leitwert to sensor LSE01 and saving to series_1_sausage_3.
Converting measurement Wurst 3 Temperatur to sensor LTC-FS and saving to series_1_sausage_3.
Converting measurement Wurst 4 Feuchtigkeit, Temperatur & Leitwert to sensor LSE01 and saving to series_1_sausage_4.
Converting measurement Wurst 4 Temperatur to sensor LTC-FS and saving to series_1_sausage_4.
Converting measurement Wurst 5 Feuchtigkeit, Temperatur & Leitwert to sensor LSE01 and saving to series_1_

{'series_1_sausage_1':                        Gemessen am  Gerät  Environment Humdity  \
 0      2023-08-02 23:56:34.268438Z  LSE01                  NaN   
 1      2023-08-02 23:36:34.335206Z  LSE01                  NaN   
 2      2023-08-02 23:16:34.367677Z  LSE01                  NaN   
 3      2023-08-02 22:56:34.128047Z  LSE01                  NaN   
 4      2023-08-02 22:36:34.443900Z  LSE01                  NaN   
 ...                            ...    ...                  ...   
 86232  2023-07-31 22:31:36.026797Z  WS202                  NaN   
 86233  2023-07-31 22:31:36.026797Z  WS202                  NaN   
 86234  2023-07-31 22:01:36.002639Z  WS202                  NaN   
 86235  2023-07-31 22:01:36.002639Z  WS202                  NaN   
 86236  2023-07-31 22:01:36.002639Z  WS202                  NaN   
 
       Environment Light status Environment Pir status  Sausage conductivity  \
 0                          NaN                    NaN                  22.0   
 1          

In [61]:
column_mappings = {
    "gerät": "sensor", 
    "gemessen_am": "timestamp", 
    "annotation": "annotation",
    "environment_light_status":"WS202_light_status",
    "light_status":"WS202_light_status",
    "environment_pir_status":"WS202_pir_status",
    "pir_status":"WS202_pir_status",
    "sausage_conductivity":"LSE01_conductivity",
    "soil_conductivity":"LSE01_conductivity",
    "sausage_moisture":"LSE01_conductivity",
    "soil_moisture":"LSE01_conductivity",
    "sausage_temperature_sensor_2":"LTC-FS_temperature",
    "temp1":"LTC-FS_temperature",
    "sausage_temperature_sensor_3":"LSE01_temperature",
    "soil_temperature":"LSE01_temperature",
}
annotations_mapping = {
    "Umroetung": "Curing",
    "Reifung Phase 1": "Ripening Phase 1",
    "Reifung Phase 2": "Ripening Phase 2"
}

In [76]:
for file_name, file_df in file_dfs.items():
    if file_df.empty:
        print(f"No data in {file_name} from {sheet} in {file}. Skipping file conversion.")
        continue
    file_df.rename(columns=lambda col: col.strip().replace(" ", "_").lower(), inplace=True)
    file_df.rename(columns=column_mappings, inplace=True)
    if "annotation" in file_df.columns:
        file_df.dropna(subset=["annotation"], inplace=True)
        for annotation_value, annotation_label in annotations_mapping.items():
            file_df["annotation"] = file_df["annotation"].replace(annotation_value, annotation_label)
    file_df["timestamp"] = pd.to_datetime(file_df["timestamp"], errors="coerce")
    file_df.sort_values(by="timestamp", inplace=True)
    file_df.dropna(axis=1, how="all", inplace=True)
    if file_df.columns.duplicated().any():
        print(f"Warning: Duplicate columns found in {file_name} after processing. Columns: {file_df.columns[file_df.columns.duplicated()]}")

file_dfs

{'series_1_sausage_1':                              timestamp  sensor ws202_light_status  \
 9024  2023-03-30 00:01:39.515132+00:00   LSE01                NaN   
 41461 2023-03-30 00:03:38.046627+00:00   WS202               dark   
 41459 2023-03-30 00:03:38.046627+00:00   WS202                NaN   
 41460 2023-03-30 00:03:38.046627+00:00   WS202                NaN   
 18055        2023-03-30 00:10:33+00:00  LTC-FS                NaN   
 ...                                ...     ...                ...   
 18056 2023-08-02 23:31:37.130589+00:00   WS202                NaN   
 18058 2023-08-02 23:31:37.130589+00:00   WS202                NaN   
 1     2023-08-02 23:36:34.335206+00:00   LSE01                NaN   
 9025         2023-08-02 23:46:14+00:00  LTC-FS                NaN   
 0     2023-08-02 23:56:34.268438+00:00   LSE01                NaN   
 
       ws202_pir_status  lse01_conductivity  lse01_conductivity  \
 9024               NaN              3851.0                1.46   
 4

In [73]:
for file_name, file_df in file_dfs.items():
    if file_df.empty:
        print(f"No data to save for {file_name} from {sheet} in {file}. Skipping file creation.")
        continue
    output_path = os.path.join(converted_data_path, f"{file_name}.csv")
    print(f"Saving converted data to {output_path}...")
    file_df.to_csv(output_path, index=False)
    print(f"Saved converted data to {output_path} successfully.")

Saving converted data to f:\_Data\Uni\MA_Project\masterproject-lorawan-demonstrator\data-preperation\..\data\training\converted\series_1_sausage_1.csv...
Saved converted data to f:\_Data\Uni\MA_Project\masterproject-lorawan-demonstrator\data-preperation\..\data\training\converted\series_1_sausage_1.csv successfully.
Saving converted data to f:\_Data\Uni\MA_Project\masterproject-lorawan-demonstrator\data-preperation\..\data\training\converted\series_1_sausage_2.csv...
Saved converted data to f:\_Data\Uni\MA_Project\masterproject-lorawan-demonstrator\data-preperation\..\data\training\converted\series_1_sausage_2.csv successfully.
Saving converted data to f:\_Data\Uni\MA_Project\masterproject-lorawan-demonstrator\data-preperation\..\data\training\converted\series_1_sausage_3.csv...
Saved converted data to f:\_Data\Uni\MA_Project\masterproject-lorawan-demonstrator\data-preperation\..\data\training\converted\series_1_sausage_3.csv successfully.
Saving converted data to f:\_Data\Uni\MA_Proje

In [77]:
for workbook in excel_files:
    workbook[1].close()